# 强化学习与大模型后训练 · 第 3/12 课：策略梯度与基线

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：从 log-derivative trick 解释 REINFORCE，并实现不改变期望梯度的 baseline advantage。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、概率期望、基本深度学习
- 本课在路线中的作用：策略梯度直接提高高回报动作的对数概率；baseline 只降低方差，不应依赖当前采样动作。

## 核心心智模型

### 1. 用回报给 log-prob 梯度定方向

J(θ)=E[R]，策略梯度为 E[∇logπ(a|s)(G-b(s))]。baseline 与当前动作独立时，E[b(s)∇logπ(a|s)]=0，因此只改变估计方差。实际最小化负号损失，return/advantage 应 stop-gradient；无需也不能把普通反向传播穿过离散采样动作。

### 2. 完整回答的 REINFORCE

单轮结果奖励下，L=−meanᵢ[Aᵢ Σₜ mᵢₜ logπθ(yᵢₜ|xᵢ,yᵢ,<ₜ)]。m 只选择模型生成的回答 token，排除 prompt/padding。序列 log-prob 是求和；额外除回答长度会重加权样本，并非无影响的实现细节。

### 3. RLOO：不让样本给自己做 baseline

同一 prompt 独立采样 K≥2 个回答，bᵢ=Σⱼ≠ᵢRⱼ/(K−1)，Aᵢ=Rᵢ−bᵢ。不训练 critic，但要支付多回答采样成本。若用包含自身的组均值，Aᵢ=(K−1)/K × Aᵢ,RLOO；未再做标准化时会缩小期望梯度，不能称其完全无偏。

### 4. 边界与取舍

RLOO 的无偏 baseline 论证要求条件于 prompt 的独立采样；复制同一个回答不是 K 个独立样本。学到的 critic 则用额外训练换取 token 级价值估计，后续在 GAE/PPO 中使用。

## 具体演示

同 prompt 奖励 [3,1,2]：RLOO baseline 为 [1.5,2.5,2]，优势 [1.5,−1.5,0]；包含自身的均值 baseline 得 [1,−1,0]，正好小了 2/3。

若回答两个 token 的 log-prob 为 [−0.2,−0.7]，则序列 log-prob 是 −0.9，不是 −0.45。练习的 log_probs 输入可表示动作或完整序列的 log-prob，returns 表示已冻结的学习信号。

## 实践任务：唯一代码填空题

补齐 REINFORCE 的单批损失；输入是已采样动作的 log_prob 和 detached return。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
def reinforce_loss(log_probs, returns):
    """log_probs 为动作/序列 log-prob，returns 为冻结的优势或回报。"""
    if not log_probs or len(log_probs) != len(returns):
        raise ValueError("need nonempty aligned samples")
    # TODO：只补齐下面这个表达式。
    return ______

assert reinforce_loss([-0.2, -0.7], [2.0, 1.0]) == 0.55

# 已给出的 RLOO baseline；唯一填空仍是上面的 REINFORCE 损失。
def rloo_advantages(rewards):
    if len(rewards) < 2:
        raise ValueError("RLOO needs at least two independent completions")
    total = sum(rewards)
    return [r - (total - r) / (len(rewards) - 1) for r in rewards]

assert rloo_advantages([3., 1., 2.]) == [1.5, -1.5, 0.]
assert rloo_advantages([2., 2.]) == [0., 0.]
centered = [r - 2. for r in [3., 1., 2.]]
assert all(abs(a - 2/3*b) < 1e-12
           for a, b in zip(centered, rloo_advantages([3., 1., 2.])))
assert abs(reinforce_loss([-.9], [2.]) - 1.8) < 1e-12
assert reinforce_loss([-.9], [0.]) == 0.
for bad in [[], [1.]]:
    try:
        rloo_advantages(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("undersized RLOO group must fail")
for logps, returns in [([], []), ([1.], [])]:
    try:
        reinforce_loss(logps, returns)
    except ValueError:
        pass
    else:
        raise AssertionError("unaligned or empty batch must fail")


### 检查方法

运行本单元格；所有 `assert` 必须通过。另手工构造一个边界输入，解释预期结果。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“策略梯度与基线”的工作机制。

**你的答案：**


### Q2

若直接把概率 π 而不是 logπ 乘回报，梯度估计哪里变了？

**你的答案：**


### Q3

共享 backbone 的 actor-critic 中，如何检查两个 loss 的梯度是否互相破坏？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
def reinforce_loss(log_probs, returns):
    """log_probs 为动作/序列 log-prob，returns 为冻结的优势或回报。"""
    if not log_probs or len(log_probs) != len(returns):
        raise ValueError("need nonempty aligned samples")
    # 参考实现：表达式直接对应上文不变量。
    return -sum(lp * ret for lp, ret in zip(log_probs, returns)) / len(log_probs)

assert reinforce_loss([-0.2, -0.7], [2.0, 1.0]) == 0.55

# 已给出的 RLOO baseline；唯一填空仍是上面的 REINFORCE 损失。
def rloo_advantages(rewards):
    if len(rewards) < 2:
        raise ValueError("RLOO needs at least two independent completions")
    total = sum(rewards)
    return [r - (total - r) / (len(rewards) - 1) for r in rewards]

assert rloo_advantages([3., 1., 2.]) == [1.5, -1.5, 0.]
assert rloo_advantages([2., 2.]) == [0., 0.]
centered = [r - 2. for r in [3., 1., 2.]]
assert all(abs(a - 2/3*b) < 1e-12
           for a, b in zip(centered, rloo_advantages([3., 1., 2.])))
assert abs(reinforce_loss([-.9], [2.]) - 1.8) < 1e-12
assert reinforce_loss([-.9], [0.]) == 0.
for bad in [[], [1.]]:
    try:
        rloo_advantages(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("undersized RLOO group must fail")
for logps, returns in [([], []), ([1.], [])]:
    try:
        reinforce_loss(logps, returns)
    except ValueError:
        pass
    else:
        raise AssertionError("unaligned or empty batch must fail")


### Q1 参考答案

∇J=E[∇logπ(a|s)(G-b(s))]；因为对动作分布求和时 E[∇logπ]=0，状态基线不改变期望。

### Q2 参考答案

∇π=π∇logπ。对按 π 采样的动作再用 π 乘回报，会额外乘一次动作概率，过度偏向已经常见的动作；它不是 REINFORCE 对 E[R] 的无偏梯度估计。

### Q3 参考答案

先分别取得 policy loss 和 value loss 对共享参数的梯度，比较范数与余弦相似度；持续负余弦提示方向冲突。再用相同样本做 loss 权重或独立 critic 的对照，观察策略指标和价值误差。advantage 必须 detach，但 value loss 仍可按设计更新共享 backbone。

## 参考资料

- [Back to Basics / RLOO（§2.2–2.3）](https://arxiv.org/html/2402.14740v1)

论文实验结论有具体模型和任务范围，不据此断言 RLOO 总优于 PPO。